## 1. Conexão com o SQL Server

Nesta etapa é configurada a conexão entre o Databricks e o Azure SQL Server por meio de JDBC.

As credenciais de acesso são recuperadas de forma segura a partir de um Databricks Secret Scope, evitando a exposição de informações sensíveis no código.

Essa conexão será utilizada posteriormente para gravar os dados da tabela `ecommerce_pedidos`, provenientes da camada Bronze do Data Lake, no schema `squad1` do SQL Server.

## 2. Autenticação e carregamento das credenciais

O notebook de autenticação é executado para disponibilizar as credenciais necessárias para acesso ao Azure Data Lake.

Em seguida, as credenciais do Azure SQL Server são recuperadas de forma segura por meio do Databricks Secret Scope.

In [0]:
%run ./00_autenticação

## 3. Validação das credenciais

Nesta etapa é verificado se as credenciais necessárias para conexão com o SQL Server foram carregadas corretamente, sem exibir seus valores.

In [0]:
print("client_id carregado:", "client_id" in globals())
print("client_secret carregado:", "client_secret" in globals())
print("tenant_id carregado:", "tenant_id" in globals())

In [0]:
jdbc_hostname = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_hostname"
)

jdbc_database = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_database"
)

jdbc_username = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_username"
)

jdbc_password = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_password"
)

In [0]:
print("JDBC Hostname carregado:", jdbc_hostname is not None)
print("JDBC Database carregado:", jdbc_database is not None)
print("JDBC Username carregado:", jdbc_username is not None)
print("JDBC Password carregado:", jdbc_password is not None)

## 4. Configuração da conexão JDBC

Nesta etapa são definidas a URL de conexão com o Azure SQL Server e as propriedades JDBC utilizadas pelo Spark para acessar o banco de dados.

In [0]:
jdbc_url = f"jdbc:sqlserver://{jdbc_hostname};databaseName={jdbc_database}"

In [0]:
connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


## 5. Teste de conexão com o SQL Server

É executada uma consulta simples para validar se o Databricks consegue se comunicar com o SQL Server utilizando as credenciais e configurações definidas anteriormente.

In [0]:
teste_conexao = spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT 1 AS conexao) AS teste",
    properties=connection_properties
)

display(teste_conexao)

In [0]:
# =========================================================
# 6. LEITURA DA BRONZE - ECOMMERCE_PEDIDOS
# =========================================================
# Origem: Azure Data Lake Gen2
# Camada: Bronze
# Objetivo: carregar os dados de negócio que serão
# disponibilizados no SQL Server.
#
# Nesta etapa não há gravação no banco.
# =========================================================

storage_account = "internshipdatalake"
lake_container = "squad1"

bronze_path = (
    f"abfss://{lake_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "bronze/ecommerce_pedidos"
)

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}


# Seleciona somente as colunas de negócio.
# Metadados técnicos da Bronze permanecem no Data Lake.

df_bronze_sql = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
    .select(
        "id_pedido",
        "id_cliente",
        "id_endereco_entrega",
        "dt_pedido",
        "status_pedido",
        "valor_total",
        "valor_frete",
        "metodo_pagamento",
        "dt_previsao_entrega",
        "dt_ultima_atualizacao_status"
    )
)

print("Registros preparados para o SQL:", df_bronze_sql.count())
df_bronze_sql.printSchema()

In [0]:
# =========================================================
# 7. PREPARAÇÃO DOS DADOS PARA O SQL SERVER
# =========================================================
# Objetivo:
# preparar os dados da camada Bronze antes da gravação
# na tabela squad1.ecommerce_pedidos.
#
# As colunas de data estão armazenadas na Bronze como long
# em nanossegundos. Nesta etapa, elas são convertidas para
# timestamp, formato adequado para armazenamento no SQL Server.
#
# Esta etapa ainda não grava dados no banco.
# =========================================================

from pyspark.sql import functions as F

df_pedidos_para_sql = (
    df_bronze_sql

# Converte os campos temporais de nanossegundos para timestamp.

    .withColumn(
        "dt_pedido",
        F.timestamp_micros(
            (F.col("dt_pedido") / 1000).cast("long")
        )
    )

    .withColumn(
        "dt_previsao_entrega",
        F.timestamp_micros(
            (F.col("dt_previsao_entrega") / 1000).cast("long")
        )
    )

    .withColumn(
        "dt_ultima_atualizacao_status",
        F.timestamp_micros(
            (F.col("dt_ultima_atualizacao_status") / 1000).cast("long")
        )
    )

    # Mantém somente as colunas de negócio que serão enviadas ao SQL Server.

    .select(
        "id_pedido",
        "id_cliente",
        "id_endereco_entrega",
        "dt_pedido",
        "status_pedido",
        "valor_total",
        "valor_frete",
        "metodo_pagamento",
        "dt_previsao_entrega",
        "dt_ultima_atualizacao_status"
    )
)

print("Registros preparados:", df_pedidos_para_sql.count())

df_pedidos_para_sql.printSchema()

display(df_pedidos_para_sql.limit(5))

In [0]:
# =========================================================
# 8. GRAVAÇÃO NO AZURE SQL SERVER
# =========================================================
# Destino: squad1.ecommerce_pedidos
#
# O DataFrame preparado na etapa anterior é gravado
# no Azure SQL Server.
#
# O modo overwrite é utilizado para manter a tabela
# sincronizada com o estado atual completo da Bronze,
# evitando duplicidade em reprocessamentos.
# =========================================================

sql_table = "squad1.ecommerce_pedidos"

(
    df_pedidos_para_sql
    # Grava os dados preparados na tabela da dupla no schema squad1.
    .write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_hostname)
    .option("database", jdbc_database)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("authentication", "SqlPassword")
    .option("encrypt", "true")
    .option("dbtable", sql_table)
    .save()
)

print("✅ squad1.ecommerce_pedidos gravada no Azure SQL.")

In [0]:
# =========================================================
# 9. VALIDAÇÃO DA GRAVAÇÃO NO AZURE SQL SERVER
# =========================================================
# Objetivo:
# confirmar que os dados gravados no SQL Server possuem
# a mesma quantidade de registros preparados a partir
# da camada Bronze.
#
# A tabela é lida novamente do banco para validar
# contagem e estrutura após a gravação.
# =========================================================

df_sql_pedidos = spark.read.jdbc(
    url=jdbc_url,
    table="squad1.ecommerce_pedidos",
    properties=connection_properties
)

qtd_bronze = df_pedidos_para_sql.count()
qtd_sql = df_sql_pedidos.count()

print("Registros preparados da Bronze:", qtd_bronze)
print("Registros gravados no SQL:", qtd_sql)

# Interrompe a execução caso exista divergência de quantidade.
assert qtd_bronze == qtd_sql, \
    "❌ Divergência entre Bronze e SQL Server."

print("\nSchema da tabela no SQL:")
df_sql_pedidos.printSchema()

print("\n✅ Validação concluída: Bronze e SQL possuem a mesma quantidade de registros.")